task 3. build a small neural network using Tenserflow.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

#generate random data 
np.random.seed(42)
n_samples = 200

#Temparature (170-250°C)
temperature = np.random.uniform(170, 250, n_samples)

#Time (8-20 min)
time = np.random.uniform(8, 20, n_samples)

#1 - if good roasting
#rule : duration is best kept between 12 and 15 minutes while the temp should be between 175 and 250 degrees Celsius
y = ((temperature > 175) & (temperature < 250) & 
     (time > 12) & (time < 15)).astype(int)

X = np.column_stack([temperature, time])  #2 simple features

In [2]:
X
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1,
       0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1,
       0, 0])

In [ ]:
print(X.shape)
print(y.shape)

(200, 2)
(200,)


In [ ]:
import matplotlib.pyplot as plt

scatter = plt.scatter(X[:, 0], X[:, 1],c=y,cmap='bwr')
plt.xlabel('Temperature (°C)')
plt.ylabel('Time (minutes)')
plt.title('Coffee Roasting Dataset\nRed=Well-roasted, Blue=Over-roasted')
plt.grid(True,alpha=0.2)
plt.tight_layout()
plt.show()

In [3]:
#normalize data
print(f"Temperature Max, Min pre normalization: {np.max(X[:,0]):0.2f}, {np.min(X[:,0]):0.2f}")
print(f"Time Max, Min pre normalization: {np.max(X[:,1]):0.2f}, {np.min(X[:,1]):0.2f}")
norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)
Xn = norm_l(X)
print(f"Temperature Max, Min post normalization: {np.max(Xn[:,0]):0.2f}, {np.min(Xn[:,0]):0.2f}")
print(f"Time Max, Min post normalization: {np.max(Xn[:,1]):0.2f}, {np.min(Xn[:,1]):0.2f}")

Temperature Max, Min pre normalization: 248.95, 170.44
Time Max, Min pre normalization: 19.89, 8.06
Temperature Max, Min post normalization: 1.71, -1.63
Time Max, Min post normalization: 1.66, -1.71


In [7]:
#do data tiling for reduce number of epochs and increase gredient descent
Xt = np.tile(Xn,(1000,1))
Yt = np.tile(y,(1000,))
print(Xt.shape, Yt.shape)


(200000, 2) (200000,)


In [8]:
tf.random.set_seed(1234)
model = Sequential(
    [
        tf.keras.Input((2,)),
        Dense(3,activation='sigmoid',name='layer1'),
        Dense(1,activation='sigmoid',name='layer2')
    ]
)

In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ layer1 (Dense)                  │ (None, 3)              │             9 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Dense)                  │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13 (52.00 B)

 Trainable params: 13 (52.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.compile(
    loss = tf.keras.losses.BinaryCrossentropy(),
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01),
)

model.fit(
    Xt, Yt, epochs=15,
)

Epoch 1/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 2s 237us/step - loss: 0.0147
Epoch 2/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 229us/step - loss: 0.0140
Epoch 3/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 231us/step - loss: 0.0134
Epoch 4/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 234us/step - loss: 0.0129
Epoch 5/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 2s 239us/step - loss: 0.0125
Epoch 6/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 2s 242us/step - loss: 0.0121
Epoch 7/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 2s 239us/step - loss: 0.0118
Epoch 8/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 1s 236us/step - loss: 0.0115
Epoch 9/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 2s 240us/step - loss: 0.0113
Epoch 10/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 2s 244us/step - loss: 0.0111
Epoch 11/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 2s 243us/step - loss: 0.0109
Epoch 12/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 2s 246us/step - loss: 0.0108
Epoch 13/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 2s 245us/step - loss: 0.0107
Epoch 14/15
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 2s 241us/step - loss: 0.0106
E

In [14]:
#predict with new positive values
X_test = np.array([
    [200,13.9],
    [200,17]
])

X_testn = norm_l(X_test)
predictions = model.predict(X_testn)
predictions

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


array([[1.0000000e+00],
       [2.1490286e-20]], dtype=float32)

In [15]:
yhat = (predictions>=0.5).astype(int)
yhat

array([[1],
       [0]])